Building on the stochastic hazard generation framework, we aggregate the ensemble of 400 scenarios generated by the graph-aware conditional diffusion model for each return level to the ward scale. For each ward, scenario-wise hazard estimates are spatially integrated to derive a probabilistic hazard index that captures both intensity and uncertainty. This hazard index is then combined with structural vulnerability, represented through flood susceptibility and socio-environmental factors,


following the IPCC risk formulation: Risk = Hazard × Exposure × Sensitivity × (1 − Adaptive Capacity). The resulting composite flood risk index provides a spatially explicit, probabilistic assessment of climate risk at the ward level, enabling more robust and actionable decision-making compared to deterministic approaches.

In [59]:
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import box

# -----------------------------
# LOAD DATA
# -----------------------------
grid_evt = gpd.read_file(
"/vol/sandeep_storage/Files2/Hydro_Graph/hyderabad_hazard_fields.geojson"
)

wards = gpd.read_file(
"/vol/sandeep_storage/Files2/Hydro_Graph/ward_hydrographdiff_final.geojson"
).to_crs(epsg=4326)

wards["ward_id"] = wards["ward_id"].astype(str)

# -----------------------------
# GRID COORDS
# -----------------------------
lats = np.sort(grid_evt["lat"].unique())
lons = np.sort(grid_evt["lon"].unique())


In [60]:
# -----------------------------
# CREATE GRID POLYGONS
# -----------------------------
lat_res = abs(lats[1] - lats[0])
lon_res = abs(lons[1] - lons[0])

cells = []

for i, lat in enumerate(lats):
    for j, lon in enumerate(lons):

        poly = box(
            lon - lon_res/2,
            lat - lat_res/2,
            lon + lon_res/2,
            lat + lat_res/2
        )

        cells.append({
            "i": i,
            "j": j,
            "geometry": poly
        })

grid_cells = gpd.GeoDataFrame(cells, crs="EPSG:4326")


In [61]:
# -----------------------------
# INTERSECT GRID WITH WARDS
# -----------------------------
grid_ward_map = gpd.overlay(
    grid_cells,
    wards[["ward_id","geometry"]],
    how="intersection"
)

print("Overlay done:", grid_ward_map.shape)


Overlay done: (208, 4)


In [62]:
# -----------------------------
# COMPUTE AREA WEIGHTS
# -----------------------------
grid_ward_map["area"] = grid_ward_map.geometry.area

grid_ward_map["weight"] = (
    grid_ward_map["area"] /
    grid_ward_map.groupby("ward_id")["area"].transform("sum")
)

print("Weights ready")


Weights ready


/tmp/ipykernel_298547/2274652807.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  grid_ward_map["area"] = grid_ward_map.geometry.area


In [63]:
def aggregate_to_ward_weighted(scenarios, mapping):

    mapping = mapping.copy()

    mapping["i"] = mapping["i"].astype(int)
    mapping["j"] = mapping["j"].astype(int)

    results = []

    for s in scenarios:

        mapping["hazard"] = s[mapping["i"], mapping["j"]]

        mapping["weighted"] = mapping["hazard"] * mapping["weight"]

        ward_val = mapping.groupby("ward_id")["weighted"].sum()

        results.append(ward_val)

    return pd.DataFrame(results)


In [64]:
def compute_hazard_index(df_ward_scenarios):

    def normalize(x):
        return (x - x.min()) / (x.max() - x.min() + 1e-8)

    mean_hazard = df_ward_scenarios.mean(axis=0)
    p90_hazard  = df_ward_scenarios.quantile(0.9, axis=0)
    p95_hazard  = df_ward_scenarios.quantile(0.95, axis=0)
    std_hazard  = df_ward_scenarios.std(axis=0)

    mean_norm = normalize(mean_hazard)
    p90_norm  = normalize(p90_hazard)
    p95_norm  = normalize(p95_hazard)
    std_norm  = normalize(std_hazard)

    # Tail fusion
    tail = 0.6 * p90_norm + 0.4 * p95_norm

    hazard_index = (
        0.5 * mean_norm +
        0.35 * tail +
        0.15 * std_norm
    )

    hazard_index.index = hazard_index.index.astype(str)

    return hazard_index


In [65]:
hazard_results = {}

for rl in ["10","25","50","100"]:

    print(f"\nProcessing RL {rl}")

    scenarios = np.load(
        f"/vol/sandeep_storage/Files2/Hydro_Graph/cond_ddpm_outputs/cond_scenarios_rl{rl}.npy"
    )

    ward_df = aggregate_to_ward_weighted(scenarios, grid_ward_map)

    print("Ward coverage:", ward_df.shape)

    hazard_index = compute_hazard_index(ward_df)

    hazard_results[rl] = hazard_index



Processing RL 10


Ward coverage: (400, 142)

Processing RL 25
Ward coverage: (400, 142)

Processing RL 50
Ward coverage: (400, 142)

Processing RL 100
Ward coverage: (400, 142)


In [66]:
for rl in ["10","25","50","100"]:

    hazard_df = hazard_results[rl].reset_index()
    hazard_df.columns = ["ward_id", f"hazard_rl{rl}"]

    wards = wards.merge(
        hazard_df,
        on="ward_id",
        how="left"
    )


In [67]:
for rl in ["10","25","50","100"]:
    print(f"NaNs in hazard_rl{rl}:",
          wards[f"hazard_rl{rl}"].isna().sum())


NaNs in hazard_rl10: 0
NaNs in hazard_rl25: 0
NaNs in hazard_rl50: 0
NaNs in hazard_rl100: 0


In [68]:
for rl in ["10","25","50","100"]:
    wards[f"risk_rl{rl}"] = (
        wards[f"hazard_rl{rl}"] *
        wards["structural_vulnerability"]
    )


In [69]:
for rl in ["10","25","50","100"]:
    col = f"risk_rl{rl}"
    wards[f"risk_norm_rl{rl}"] = (
        (wards[col] - wards[col].min()) /
        (wards[col].max() - wards[col].min() + 1e-8)
    )


In [70]:
import os

output_dir = "/vol/sandeep_storage/Files2/Hydro_Graph/final_risk_outputs"
os.makedirs(output_dir, exist_ok=True)

# GEOJSON
wards.to_file(
    f"{output_dir}/ward_flood_risk_normalized.geojson",
    driver="GeoJSON"
)

# CSV
cols = ["ward_id"] + \
       [f"risk_norm_rl{rl}" for rl in ["10","25","50","100"]]

wards[cols].to_csv(
    f"{output_dir}/ward_flood_risk_normalized.csv",
    index=False
)

# PARQUET
wards[cols].to_parquet(
    f"{output_dir}/ward_flood_risk_normalized.parquet",
    index=False
)

print("\n FINAL PIPELINE COMPLETE")



 FINAL PIPELINE COMPLETE


In [71]:
wards


,ward_id,area_km2,road_km,road_density,dist_hospital_m,dist_hospital_km,elev_mean,slope_mean,pop_value,pop_density,...,hazard_rl50,hazard_rl100,risk_rl10,risk_rl25,risk_rl50,risk_rl100,risk_norm_rl10,risk_norm_rl25,risk_norm_rl50,risk_norm_rl100
0,relation/7848799,8.619220,116.476667,13.513598,791.286374,0.791286,502.0,0.896808,127.148150,53.072723,...,0.356905,0.372054,0.025983,0.023678,0.017462,0.018203,0.411362,0.404411,0.242366,0.320099
1,relation/7848844,21.230577,335.166191,15.786956,1176.949230,1.176949,511.0,1.828200,56.710102,2.671152,...,0.177719,0.085655,0.000206,0.000127,0.000235,0.000113,0.003265,0.002170,0.003262,0.001992
2,relation/7848845,14.171507,245.694304,17.337204,2544.524743,2.544525,614.0,2.497209,66.898666,4.720646,...,0.240020,0.373270,0.000973,0.001095,0.000945,0.001470,0.015408,0.018706,0.013120,0.025850
3,relation/7848859,8.671361,170.680433,19.683235,374.373286,0.374373,484.0,0.984000,65.211502,7.520331,...,0.304163,0.318717,0.001894,0.001734,0.001197,0.001255,0.029985,0.029608,0.016621,0.022065
4,relation/7848886,10.261469,235.856454,22.984667,692.935948,0.692936,379.0,1.020943,25.397692,2.475054,...,0.100536,0.264373,0.000110,0.000190,0.000089,0.000234,0.001747,0.003254,0.001234,0.004111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,relation/7867249,4.633468,133.400564,28.790652,46.309938,0.046310,603.0,0.896808,215.259480,46.457530,...,0.294100,0.159785,0.005599,0.005190,0.006386,0.003470,0.088640,0.088640,0.088640,0.061014
138,relation/7867252,1.471230,39.865393,27.096649,132.527164,0.132527,487.0,0.984000,281.922210,191.623529,...,0.294100,0.159785,0.032164,0.029815,0.036689,0.019933,0.509230,0.509230,0.509230,0.350520
139,relation/7867584,1.663302,39.697309,23.866563,866.686569,0.866687,505.0,1.581730,150.713200,90.610828,...,0.294100,0.159785,0.016978,0.015738,0.019367,0.010522,0.268805,0.268805,0.268805,0.185028
140,relation/7867585,1.638009,41.437681,25.297592,400.035027,0.400035,434.0,0.619390,67.966736,41.493509,...,0.277901,0.256964,0.008271,0.007576,0.005600,0.005178,0.130941,0.129396,0.077728,0.091058


In [72]:
wards.columns


Index(['ward_id', 'area_km2', 'road_km', 'road_density', 'dist_hospital_m',
       'dist_hospital_km', 'elev_mean', 'slope_mean', 'pop_value',
       'pop_density', 'dist_drainage_m', 'dist_musi_m', 'sens_floodplain',
       'impervious_frac', 'river_km', 'drain_density', 'sens_elev',
       'sens_slope', 'sens_drainage', 'sens_impervious', 'sens_drain_density',
       'exposure_index', 'ac_roads', 'ac_hospital', 'adaptive_capacity',
       'sensitivity_index', 'potential_impact', 'structural_vulnerability',
       'geometry', 'hazard_rl10', 'hazard_rl25', 'hazard_rl50', 'hazard_rl100',
       'risk_rl10', 'risk_rl25', 'risk_rl50', 'risk_rl100', 'risk_norm_rl10',
       'risk_norm_rl25', 'risk_norm_rl50', 'risk_norm_rl100'],
      dtype='object')

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box

# ============================================================
# PATHS
# ============================================================

base_dir = "/vol/sandeep_storage/Files2/Hydro_Graph"

delta_paths = {
    "245_near": f"{base_dir}/delta_miroc6_ssp245_near.parquet",
    "245_mid":  f"{base_dir}/delta_miroc6_ssp245_mid.parquet",
    "585_near": f"{base_dir}/delta_miroc6_ssp585_near.parquet",
    "585_mid":  f"{base_dir}/delta_miroc6_ssp585_mid.parquet",
}

scen_dir = f"{base_dir}/cond_ddpm_outputs"

ward_path = f"{base_dir}/ward_hydrographdiff_final.geojson"
grid_path = f"{base_dir}/hyderabad_hazard_fields.geojson"

# ============================================================
# LOAD DATA
# ============================================================

print("Loading data...")

grid_evt = gpd.read_file(grid_path)
wards = gpd.read_file(ward_path).to_crs(epsg=4326)
wards["ward_id"] = wards["ward_id"].astype(str)

# ============================================================
# GRID STRUCTURE
# ============================================================

lats = np.sort(grid_evt["lat"].unique())
lons = np.sort(grid_evt["lon"].unique())

lat_res = abs(lats[1] - lats[0])
lon_res = abs(lons[1] - lons[0])

# ============================================================
# GRID POLYGONS
# ============================================================

cells = []
for i, lat in enumerate(lats):
    for j, lon in enumerate(lons):

        poly = box(
            lon - lon_res/2,
            lat - lat_res/2,
            lon + lon_res/2,
            lat + lat_res/2
        )

        cells.append({"i": i, "j": j, "geometry": poly})

grid_cells = gpd.GeoDataFrame(cells, crs="EPSG:4326")

# ============================================================
# GRID-WARD MAPPING (FIX CRS ISSUE)
# ============================================================

grid_ward_map = gpd.overlay(
    grid_cells,
    wards[["ward_id","geometry"]],
    how="intersection"
)

#  FIX: project before area calculation
grid_ward_map = grid_ward_map.to_crs(epsg=3857)

grid_ward_map["area"] = grid_ward_map.geometry.area

grid_ward_map["weight"] = (
    grid_ward_map["area"] /
    grid_ward_map.groupby("ward_id")["area"].transform("sum")
)

print("Grid-ward mapping ready")

# ============================================================
# LOAD DELTAS
# ============================================================

print("Loading deltas...")

def load_delta(path):

    df = pd.read_parquet(path)
    value_col = [c for c in df.columns if c not in ["lat","lon"]][0]

    da = df.set_index(["lat","lon"])[value_col].to_xarray()
    da_interp = da.interp(lat=lats, lon=lons, method="linear")

    delta = np.nan_to_num(da_interp.values, nan=1.0)
    return delta

deltas = {k: load_delta(v) for k, v in delta_paths.items()}

print("Deltas loaded")

# ============================================================
# AGGREGATION
# ============================================================

def aggregate_to_ward_weighted(scenarios, mapping):

    mapping = mapping.copy()
    mapping["i"] = mapping["i"].astype(int)
    mapping["j"] = mapping["j"].astype(int)

    results = []

    for s in scenarios:
        mapping["hazard"] = s[mapping["i"], mapping["j"]]
        mapping["weighted"] = mapping["hazard"] * mapping["weight"]

        ward_val = mapping.groupby("ward_id")["weighted"].sum()
        results.append(ward_val)

    return pd.DataFrame(results)

# ============================================================
# HAZARD INDEX
# ============================================================

def compute_hazard_index(df):

    def normalize(x):
        return (x - x.min()) / (x.max() - x.min() + 1e-8)

    mean = normalize(df.mean())
    p90  = normalize(df.quantile(0.9))
    p95  = normalize(df.quantile(0.95))
    std  = normalize(df.std())

    tail = 0.6 * p90 + 0.4 * p95

    return (0.5 * mean + 0.35 * tail + 0.15 * std)

# ============================================================
# MAIN LOOP (ALL FEATURES)
# ============================================================

results = {}

for rl in ["10","25","50","100"]:

    print(f"\nProcessing RL {rl}")

    scenarios = np.load(f"{scen_dir}/cond_scenarios_rl{rl}.npy")

    #  baseline aggregation (for advanced metrics)
    ward_df_base = aggregate_to_ward_weighted(scenarios, grid_ward_map)
    ward_df_base.columns = ward_df_base.columns.astype(str)

    # ============================
    # ADVANCED METRICS (BASE SCENARIOS)
    # ============================

    # 1. hazard uncertainty
    unc = ward_df_base.std(axis=0)

    wards[f"hazard_uncertainty_rl{rl}"] = wards["ward_id"].map(unc)

    wards[f"uncertainty_norm_rl{rl}"] = (
        (wards[f"hazard_uncertainty_rl{rl}"] -
         wards[f"hazard_uncertainty_rl{rl}"].min()) /
        (wards[f"hazard_uncertainty_rl{rl}"].max() -
         wards[f"hazard_uncertainty_rl{rl}"].min() + 1e-8)
    )

    # 2. probability of extreme
    threshold = ward_df_base.mean(axis=0)
    prob = (ward_df_base > threshold).mean(axis=0)

    wards[f"prob_extreme_rl{rl}"] = wards["ward_id"].map(prob)

    # ============================
    # FUTURE SCENARIOS
    # ============================

    for key, delta in deltas.items():

        scen_scaled = scenarios * delta

        ward_vals = aggregate_to_ward_weighted(
            scen_scaled, grid_ward_map
        )

        # hazard index
        hazard_index = compute_hazard_index(ward_vals)

        # risk
        risk_vals = hazard_index * wards.set_index("ward_id")["structural_vulnerability"]

        # uncertainty (risk)
        risk_scenarios = ward_vals.multiply(
            wards.set_index("ward_id")["structural_vulnerability"],
            axis=1
        )

        risk_std = risk_scenarios.std(axis=0)

        # -------------------------
        # SAVE INTO WARDS
        # -------------------------

        wards[f"hazard_{key}_rl{rl}"] = wards["ward_id"].map(hazard_index)
        wards[f"risk_{key}_rl{rl}"] = wards["ward_id"].map(risk_vals)
        wards[f"risk_uncertainty_{key}_rl{rl}"] = wards["ward_id"].map(risk_std)

        # normalization
        col = f"risk_{key}_rl{rl}"
        wards[f"{col}_norm"] = (
            (wards[col] - wards[col].min()) /
            (wards[col].max() - wards[col].min() + 1e-8)
        )

# ============================================================
# RANKING (USE SSP585 MID AS DEFAULT STRONG SIGNAL)
# ============================================================

for rl in ["10","25","50","100"]:
    wards[f"rank_rl{rl}"] = wards[f"risk_585_mid_rl{rl}_norm"].rank(ascending=False)

print("\n FINAL PIPELINE COMPLETE (ALL FEATURES)")

# ============================================================
# SAVE EVERYTHING
# ============================================================

output_dir = f"{base_dir}/final_risk_outputs"
os.makedirs(output_dir, exist_ok=True)

# GEOJSON
wards.to_file(
    f"{output_dir}/ward_flood_risk_advanced.geojson",
    driver="GeoJSON"
)

# CSV
wards.drop(columns="geometry").to_csv(
    f"{output_dir}/ward_flood_risk_advanced.csv",
    index=False
)

# PARQUET
wards.drop(columns="geometry").to_parquet(
    f"{output_dir}/ward_flood_risk_advanced.parquet",
    index=False
)

print("\n ADVANCED DATASET SAVED SUCCESSFULLY ")


---

In [73]:
# ============================================================
# ADD ADVANCED FEATURES (UNCERTAINTY + PROBABILITY + RANK)
# ============================================================

rls = ["10","25","50","100"]

for rl in rls:

    print(f"\nAdvanced metrics for RL {rl}")

    # -----------------------------
    # LOAD SCENARIOS
    # -----------------------------
    scenarios = np.load(
        f"/vol/sandeep_storage/Files2/Hydro_Graph/cond_ddpm_outputs/cond_scenarios_rl{rl}.npy"
    )

    # -----------------------------
    # GRID → WARD (reuse your function)
    # -----------------------------
    ward_df = aggregate_to_ward_weighted(scenarios, grid_ward_map)

    # ensure alignment
    ward_df.columns = ward_df.columns.astype(str)

    # -----------------------------
    # 1️ HAZARD UNCERTAINTY
    # -----------------------------
    unc = ward_df.std(axis=0)

    wards[f"hazard_uncertainty_rl{rl}"] = wards["ward_id"].map(unc)

    wards[f"uncertainty_norm_rl{rl}"] = (
        (wards[f"hazard_uncertainty_rl{rl}"] -
         wards[f"hazard_uncertainty_rl{rl}"].min()) /
        (wards[f"hazard_uncertainty_rl{rl}"].max() -
         wards[f"hazard_uncertainty_rl{rl}"].min() + 1e-8)
    )

    # -----------------------------
    # 2️ EXTREME PROBABILITY
    # -----------------------------
    threshold = ward_df.mean(axis=0)

    prob = (ward_df > threshold).mean(axis=0)

    wards[f"prob_extreme_rl{rl}"] = wards["ward_id"].map(prob)

    # -----------------------------
    # 3️ RISK UNCERTAINTY
    # -----------------------------
    risk_scenarios = ward_df.multiply(
        wards.set_index("ward_id")["structural_vulnerability"],
        axis=1
    )

    risk_std = risk_scenarios.std(axis=0)

    wards[f"risk_uncertainty_rl{rl}"] = wards["ward_id"].map(risk_std)


# ============================================================
# 4️ RANKING
# ============================================================

for rl in rls:
    wards[f"rank_rl{rl}"] = wards[f"risk_norm_rl{rl}"].rank(ascending=False)


# ============================================================
# SAVE UPDATED DATASET
# ============================================================

output_dir = "/vol/sandeep_storage/Files2/Hydro_Graph/final_risk_outputs"

# FULL GEOJSON
wards.to_file(
    f"{output_dir}/ward_flood_risk_advanced.geojson",
    driver="GeoJSON"
)

# FULL CSV
wards.drop(columns="geometry").to_csv(
    f"{output_dir}/ward_flood_risk_advanced.csv",
    index=False
)

# FULL PARQUET
wards.drop(columns="geometry").to_parquet(
    f"{output_dir}/ward_flood_risk_advanced.parquet",
    index=False
)

print("\n ADVANCED FEATURES ADDED & SAVED")



Advanced metrics for RL 10



Advanced metrics for RL 25

Advanced metrics for RL 50

Advanced metrics for RL 100

 ADVANCED FEATURES ADDED & SAVED


In [74]:
wards


,ward_id,area_km2,road_km,road_density,dist_hospital_m,dist_hospital_km,elev_mean,slope_mean,pop_value,pop_density,...,prob_extreme_rl50,risk_uncertainty_rl50,hazard_uncertainty_rl100,uncertainty_norm_rl100,prob_extreme_rl100,risk_uncertainty_rl100,rank_rl10,rank_rl25,rank_rl50,rank_rl100
0,relation/7848799,8.619220,116.476667,13.513598,791.286374,0.791286,502.0,0.896808,127.148150,53.072723,...,0.5125,1.215691,35.502625,0.884192,0.4975,1.737000,21.0,26.0,39.0,25.0
1,relation/7848844,21.230577,335.166191,15.786956,1176.949230,1.176949,511.0,1.828200,56.710102,2.671152,...,0.4950,0.027660,26.682459,0.423370,0.5250,0.035287,140.0,141.0,137.0,141.0
2,relation/7848845,14.171507,245.694304,17.337204,2544.524743,2.544525,614.0,2.497209,66.898666,4.720646,...,0.4950,0.078169,24.939844,0.332324,0.5050,0.098216,132.0,132.0,129.0,125.0
3,relation/7848859,8.671361,170.680433,19.683235,374.373286,0.374373,484.0,0.984000,65.211502,7.520331,...,0.5150,0.091198,32.944821,0.750556,0.4925,0.129702,123.0,125.0,127.0,128.0
4,relation/7848886,10.261469,235.856454,22.984667,692.935948,0.692936,379.0,1.020943,25.397692,2.475054,...,0.5225,0.017294,27.665276,0.474718,0.4950,0.024461,141.0,140.0,141.0,140.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,relation/7867249,4.633468,133.400564,28.790652,46.309938,0.046310,603.0,0.896808,215.259480,46.457530,...,0.4900,0.549985,32.893889,0.747895,0.5050,0.714282,99.0,106.0,85.0,113.0
138,relation/7867252,1.471230,39.865393,27.096649,132.527164,0.132527,487.0,0.984000,281.922210,191.623529,...,0.4900,3.159612,32.893889,0.747895,0.5050,4.103483,12.0,15.0,11.0,21.0
139,relation/7867584,1.663302,39.697309,23.866563,866.686569,0.866687,505.0,1.581730,150.713200,90.610828,...,0.4900,1.667853,32.893889,0.747895,0.5050,2.166090,41.0,47.0,36.0,68.0
140,relation/7867585,1.638009,41.437681,25.297592,400.035027,0.400035,434.0,0.619390,67.966736,41.493509,...,0.5150,0.426339,30.297251,0.612230,0.5025,0.610538,77.0,88.0,96.0,97.0


In [75]:
wards.columns


Index(['ward_id', 'area_km2', 'road_km', 'road_density', 'dist_hospital_m',
       'dist_hospital_km', 'elev_mean', 'slope_mean', 'pop_value',
       'pop_density', 'dist_drainage_m', 'dist_musi_m', 'sens_floodplain',
       'impervious_frac', 'river_km', 'drain_density', 'sens_elev',
       'sens_slope', 'sens_drainage', 'sens_impervious', 'sens_drain_density',
       'exposure_index', 'ac_roads', 'ac_hospital', 'adaptive_capacity',
       'sensitivity_index', 'potential_impact', 'structural_vulnerability',
       'geometry', 'hazard_rl10', 'hazard_rl25', 'hazard_rl50', 'hazard_rl100',
       'risk_rl10', 'risk_rl25', 'risk_rl50', 'risk_rl100', 'risk_norm_rl10',
       'risk_norm_rl25', 'risk_norm_rl50', 'risk_norm_rl100',
       'hazard_uncertainty_rl10', 'uncertainty_norm_rl10', 'prob_extreme_rl10',
       'risk_uncertainty_rl10', 'hazard_uncertainty_rl25',
       'uncertainty_norm_rl25', 'prob_extreme_rl25', 'risk_uncertainty_rl25',
       'hazard_uncertainty_rl50', 'uncertainty_no

----

----